<a href="https://colab.research.google.com/github/Kamal-Minocha/ucs420-/blob/main/assingment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Question 1


In [11]:
import pandas as pd

roll_number_str = '1024170188'

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
]

categories = ["billing", "account", "general"]


### Generating Dynamic FAQ Entries from Roll Number

This section extracts the last two digits from your provided `roll_number_str` and generates two new FAQ entries based on the specified category logic (`digit % 3`).

In [12]:
if len(roll_number_str) < 2 or not roll_number_str.isdigit():
    raise ValueError("Please enter a valid roll number with at least two digits in the previous cell.")

last_two_digits_str = roll_number_str[-2:]
digit1 = int(last_two_digits_str[0])
digit2 = int(last_two_digits_str[1])

dynamic_entries = []

cat1_index = digit1 % 3
cat1 = categories[cat1_index]
if cat1 == "billing":
    q1 = f"What is the due date for payment with digit {digit1}?"
    a1 = f"The payment due date is the {digit1}th of each month."
    k1 = f"due date payment month {digit1}"
elif cat1 == "account":
    q1 = f"Can I change my registered email associated with digit {digit1}?"
    a1 = f"Yes, you can update your email in account settings related to digit {digit1}."
    k1 = f"email change account update {digit1}"
else:
    q1 = f"Where can I find information about {digit1}-related services?"
    a1 = f"Information about {digit1}-related services is available on our website's help section."
    k1 = f"information services help {digit1}"
dynamic_entries.append({"question": q1, "answer": a1, "keywords": k1, "category": cat1})

cat2_index = digit2 % 3
cat2 = categories[cat2_index]
if cat2 == "billing":
    q2 = f"How can I get a refund for a {digit2}-related transaction?"
    a2 = f"Refunds for {digit2}-related transactions can be processed by contacting support."
    k2 = f"refund transaction support {digit2}"
elif cat2 == "account":
    q2 = f"What if I forget my username involving digit {digit2}?"
    a2 = f"You can retrieve your username using the 'Forgot Username' option, often tied to digit {digit2}."
    k2 = f"username forgot retrieve {digit2}"
else:
    q2 = f"Are there any new features related to digit {digit2}?"
    a2 = f"Yes, we recently launched new features involving digit {digit2}; check our announcements."
    k2 = f"new features updates announcements {digit2}"
dynamic_entries.append({"question": q2, "answer": a2, "keywords": k2, "category": cat2})

print("Generated Dynamic Entries:")
for entry in dynamic_entries:
    print(entry)


Generated Dynamic Entries:
{'question': 'Where can I find information about 8-related services?', 'answer': "Information about 8-related services is available on our website's help section.", 'keywords': 'information services help 8', 'category': 'general'}
{'question': 'Are there any new features related to digit 8?', 'answer': 'Yes, we recently launched new features involving digit 8; check our announcements.', 'keywords': 'new features updates announcements 8', 'category': 'general'}


### Final Knowledge Base DataFrame

This combines the fixed and dynamically generated FAQ entries into a single pandas DataFrame and displays the result.

In [13]:
all_entries = fixed_entries + dynamic_entries

df_faq = pd.DataFrame(all_entries)

print("\nFinal FAQ DataFrame:")
display(df_faq)



Final FAQ DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,Where can I find information about 8-related s...,Information about 8-related services is availa...,information services help 8,general
5,Are there any new features related to digit 8?,"Yes, we recently launched new features involvi...",new features updates announcements 8,general


## Question 2

In [14]:
import re

def score_faq_entries(query, df):
    """
    Scores FAQ entries based on keyword matches with the query string.

    Args:
        query (str): The user's query string.
        df (pd.DataFrame): The DataFrame containing FAQ entries.

    Returns:
        pd.DataFrame: The DataFrame with an added 'confidence_score' column,
                      sorted by confidence in descending order.
    """
    query_keywords = set(re.findall(r'\b\w+\b', query.lower()))

    def calculate_score(row):
        combined_text = (str(row['question']) + ' ' +
                         str(row['answer']) + ' ' +
                         str(row['keywords'])).lower()

        combined_keywords = set(re.findall(r'\b\w+\b', combined_text))

        # Count common keywords
        score = len(query_keywords.intersection(combined_keywords))
        return score

    df_scored = df.copy()
    df_scored['confidence_score'] = df_scored.apply(calculate_score, axis=1)

    # Filter out entries with 0 confidence and sort
    df_scored = df_scored[df_scored['confidence_score'] > 0]
    df_scored = df_scored.sort_values(by='confidence_score', ascending=False).reset_index(drop=True)

    return df_scored

# --- Demonstration ---
# Example Query 1
query1 = "how to pay my fees"
print(f"\nScoring results for query: '{query1}'")
scored_df1 = score_faq_entries(query1, df_faq)
display(scored_df1)

# Example Query 2
query2 = "reset my password please"
print(f"\nScoring results for query: '{query2}'")
scored_df2 = score_faq_entries(query2, df_faq)
display(scored_df2)

# Example Query 3 - using keywords from dynamic entries
query3 = "information about 8 services"
print(f"\nScoring results for query: '{query3}'")
scored_df3 = score_faq_entries(query3, df_faq)
display(scored_df3)



Scoring results for query: 'how to pay my fees'


,question,answer,keywords,category,confidence_score
0,how to reset password,Go to Settings > Reset Password.,password reset login,account,2
1,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,2
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general,1
3,Are there any new features related to digit 8?,"Yes, we recently launched new features involvi...",new features updates announcements 8,general,1



Scoring results for query: 'reset my password please'


,question,answer,keywords,category,confidence_score
0,how to reset password,Go to Settings > Reset Password.,password reset login,account,2



Scoring results for query: 'information about 8 services'


,question,answer,keywords,category,confidence_score
0,Where can I find information about 8-related s...,Information about 8-related services is availa...,information services help 8,general,4
1,Are there any new features related to digit 8?,"Yes, we recently launched new features involvi...",new features updates announcements 8,general,1


In [15]:
!pip install -q pypdf
import pypdf
try:
    reader = pypdf.PdfReader('/content/UCS420_Assignment4.pdf')
    print(f'Total Pages: {len(reader.pages)}')
    for i, page in enumerate(reader.pages):
        print(f'--- Page {i+1} ---')
        print(page.extract_text())
except Exception as e:
    print('Error reading PDF:', e)

Total Pages: 1
--- Page 1 ---
UCS420: Cognitive Computing 
Assignment 4 – A Cognitive FAQ System Using Pandas (Nova 2.0) 
 
Pre-requisites: Pandas DataFrame  
Q1: Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas 
DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your 
own roll number digits as follows:  
• Take the LAST TWO DIGITS of your roll number. For each digit d, compute category = ["billing", "account", 
"general"][d % 3]. Invent one realistic question+answer+3 keywords per entry that fits the assigned category (e.g. 
if d%3 gives "account", write a question like “how do I update my registered mobile number”).  
• # Example roll number ...23 -> digits 2, 3 
• # digit 2 -> category[2 % 3] = general 
• # digit 3 -> category[3 % 3] = billing 
 
Output: Print your final 6-row DataFrame. 
import pandas as pd 
 
fixed_entries = [ 
    {"question": "what is the annua

## Question 3

In [16]:
def same_category(category_name, df):
    """
    Returns all questions belonging to a given category.
    """
    matching_questions = df[df['category'].str.lower() == category_name.lower()]['question']
    return matching_questions.tolist()

chosen_category = cat1 # 'general'
questions_in_category = same_category(chosen_category, df_faq)

print(f"Questions belonging to category '{chosen_category}':")
for idx, q in enumerate(questions_in_category, 1):
    print(f"{idx}. {q}")

Questions belonging to category 'general':
1. what are your working hours
2. Where can I find information about 8-related services?
3. Are there any new features related to digit 8?


## Question 4

In [17]:
entry_idx = 0
print("Original Entry:")
print(df_faq.iloc[entry_idx])

new_keyword = input("Enter a new keyword to add to this entry: ") or "discount"

existing_keywords = df_faq.loc[entry_idx, 'keywords']
if new_keyword not in existing_keywords:
    df_faq.loc[entry_idx, 'keywords'] = f"{existing_keywords} {new_keyword}"

print("\nUpdated Entry:")
print(df_faq.iloc[entry_idx])

csv_filename = f"{roll_number_str}_faq_data.csv"
df_faq.to_csv(csv_filename, index=False)
print(f"\nSuccessfully saved the updated FAQ DataFrame to {csv_filename}!")

Original Entry:
question       what is the annual fee
answer      The annual fee is Rs 500.
keywords        fee cost price charge
category                      billing
Name: 0, dtype: object
Enter a new keyword to add to this entry: hii

Updated Entry:
question       what is the annual fee
answer      The annual fee is Rs 500.
keywords    fee cost price charge hii
category                      billing
Name: 0, dtype: object

Successfully saved the updated FAQ DataFrame to 1024170188_faq_data.csv!


## Question 5

In [18]:
category_counts = df_faq.groupby('category').size().reset_index(name='count')
print("FAQ Entries count per category:")
display(category_counts)

FAQ Entries count per category:


,category,count
0,account,1
1,billing,2
2,general,3


## Question 6

In [19]:
def score_faq_entries_v2(query, df):
    """
    Scores FAQ entries based on keyword matches with the query string.
    If two or more entries tie for the highest score, it prints all tied matching entries.
    """
    query_keywords = set(re.findall(r'\b\w+\b', query.lower()))

    def calculate_score(row):
        combined_text = (str(row['question']) + ' ' +
                         str(row['answer']) + ' ' +
                         str(row['keywords'])).lower()
        combined_keywords = set(re.findall(r'\b\w+\b', combined_text))
        return len(query_keywords.intersection(combined_keywords))

    df_scored = df.copy()
    df_scored['confidence_score'] = df_scored.apply(calculate_score, axis=1)

    df_scored = df_scored[df_scored['confidence_score'] > 0]

    if df_scored.empty:
        print("No matching FAQ entries found.")
        return df_scored

    df_scored = df_scored.sort_values(by='confidence_score', ascending=False).reset_index(drop=True)

    max_score = df_scored['confidence_score'].max()
    tied_entries = df_scored[df_scored['confidence_score'] == max_score]

    if len(tied_entries) > 1:
        print(f"\n[TIE DETECTED] There is a tie with the highest confidence score of {max_score}!")
        print("Displaying all tied matching entries:")
        display(tied_entries)
    else:
        print(f"\nNo tie detected. Single best match found with score {max_score}.")
        display(df_scored.head(1))

    return df_scored

tie_query = "how to pay fee"
print(f"Executing search for tie query: '{tie_query}'")
_ = score_faq_entries_v2(tie_query, df_faq)

single_winner_query = "reset password"
print(f"\nExecuting search for single winner query: '{single_winner_query}'")
_ = score_faq_entries_v2(single_winner_query, df_faq)

Executing search for tie query: 'how to pay fee'

No tie detected. Single best match found with score 3.


,question,answer,keywords,category,confidence_score
0,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,3



Executing search for single winner query: 'reset password'

No tie detected. Single best match found with score 2.


,question,answer,keywords,category,confidence_score
0,how to reset password,Go to Settings > Reset Password.,password reset login,account,2
